In [ ]:
import requests
import time
import pandas as pd

In [2]:
# Set file parameters
download_path = '../data/bronze/Steam/'
app_list_old = 'steam_spy_id_name.csv'
app_list = 'steam_spy_id_name_new.csv'
diff_name = 'steam_spy_id_name_diff.csv'

In [3]:
def request(url, params=None, count = 0):
    # maximum repeat 5 times
    if count > 5:
        return []
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error during the request: {e}")
        time.sleep(1)
        return request(url, params, count+1)
    
    if response.status_code == 200:
        return response.json()
    elif response == None:
        print("No answer, Trying again in 10 seconds...")
        time.sleep(1)
        return request(url, params, count+1)
    else:
        print(f"Error in the API response: {response.status_code} - Parameters: {params}")
        return None

In [4]:
def get_app_data(start, stop, parser, pause):
    """
    Return list of app data generated from parser.
    
    parser : function to handle request
    """
    app_data = []
    
    try:
        app_list_file = pd.read_csv(download_path + app_list)
    except FileNotFoundError:
        print("File 'id_name.csv' not found.")
        return 
    except pd.errors.EmptyDataError:
        print("File 'id_name.csv' is empty. Check file contents.")
        return

    # iterate through each row of app_list, confined by start and stop
    for index, row in app_list_file[start:stop].iterrows():
        print('Current index: {}'.format(index), end='\r')
        
        appid = row['appid']
        name = row['name']

        # retrive app data for a row, handled by supplied parser, and append to list
        data = parser(appid, name)
        app_data.append(data)

    time.sleep(pause) # prevent overloading api with requests
    
    return app_data

In [5]:
# Get steam spy data -- all columns
url = "https://steamspy.com/api.php"
page = 0
params = {"request": "all", "page": page}

all_data_list = []

while page < 1000:
        data = request(url, params=params, count=0)

        all_data_list = pd.concat([pd.DataFrame.from_dict(data, orient='index'), pd.DataFrame(all_data_list)], ignore_index=True)

        page += 1
        params = {"request": "all", "page": page}

        # SteamSpy offers 1000 apps per all request, should it provide less than 1000, it means that we reached the end
        if len(data) < 1000 and len(data) != 0:
            print("Less than 1000 appids found, wrapping up.")
            break

        time.sleep(1)  # 1 second delay so we don't overload the server
        print(f"Going to page {page}...")

steam_spy_all = pd.DataFrame(all_data_list)
# Get steam spy data -- only with id and name, for the use of search in steam api
games = steam_spy_all[["appid", "name"]].sort_values(by = 'appid').reset_index(drop=True)
games.to_csv(download_path + app_list)

Going to page 1...
Going to page 2...
Going to page 3...
Going to page 4...
Going to page 5...
Going to page 6...
Going to page 7...
Going to page 8...
Going to page 9...
Going to page 10...
Going to page 11...
Going to page 12...
Going to page 13...
Going to page 14...
Going to page 15...
Going to page 16...
Going to page 17...
Going to page 18...
Going to page 19...
Going to page 20...
Going to page 21...
Going to page 22...
Going to page 23...
Going to page 24...
Going to page 25...
Going to page 26...
Going to page 27...
Going to page 28...
Going to page 29...
Going to page 30...
Going to page 31...
Going to page 32...
Going to page 33...
Going to page 34...
Going to page 35...
Going to page 36...
Going to page 37...
Going to page 38...
Going to page 39...
Going to page 40...
Going to page 41...
Going to page 42...
Going to page 43...
Going to page 44...
Going to page 45...
Going to page 46...
Going to page 47...
Going to page 48...
Going to page 49...
Going to page 50...
Going to 

In [6]:
# find the different between old and new list
old_games = pd.read_csv(download_path + app_list_old)
key = "appid"
diff = games[~games[key].isin(old_games[key])]
diff.to_csv(download_path + diff_name)